<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

# Maestría en Inteligencia Artificial
## Proyecto de Innovación III
### Detección de pistolas y cuchillos en actos delictivos mediante visión computacional
---

**Integrantes:**
- Johan Sebastian Bonilla
- Edwin Gómez

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/Weapon-Detection/blob/main/notebooks/03_app.ipynb)

---

# 03 — Interfaz visual (Gradio)

Visual interactivo que permite usar el sistema completo sin tocar código:

| Pestaña | Función |
|---|---|
| 🖼️ Imagen | Sube una foto, ve las detecciones y la descripción Gen AI |
| 🎬 Video | Sube un video, descárgalo anotado con todas las detecciones |
| 📊 Métricas | Resumen de confianza y clases detectadas en la sesión |

> **Prerrequisito:** Tener `best.pt` disponible (entrenado en `01_eda.ipynb` o descargado desde Kaggle).


## 0. Entorno e instalación

In [48]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_PATH = "/content"
else:
    BASE_PATH = os.getcwd()

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"BASE_PATH: {BASE_PATH}")


Entorno: Google Colab
BASE_PATH: /content


In [49]:
# Instalar dependencias
if IN_COLAB:
    !wget -q https://raw.githubusercontent.com/sebastianb92/Weapon-Detection/main/requirements.txt -O requirements.txt
    !uv pip install -r requirements.txt
    !wget -q https://raw.githubusercontent.com/sebastianb92/Weapon-Detection/main/model/best.pt
else:
    print("Entorno local — instala con: pip install -r requirements.txt")


Using Python 3.12.13 environment at: /usr
Checked 6 packages in 96ms


## 1. Imports y configuración global

In [50]:
import cv2
import torch
import numpy as np
import gradio as gr
import json
import base64
import tempfile
import time
from pathlib import Path
from ultralytics import YOLO
from groq import Groq

print(f"Gradio  : {gr.__version__}")


Gradio  : 5.50.0


In [51]:
# ============================================================
# CONFIGURACIÓN — ajusta según tu entorno
# ============================================================

# Ruta al modelo entrenado
if IN_COLAB:
    MODEL_PATH = Path("/content/best.pt")
else:
    MODEL_PATH = Path(BASE_PATH) / "weapons_training/yolo_weapons/weights/best.pt"

# Umbral de confianza por defecto (el usuario lo puede mover en la app)
DEFAULT_CONF = 0.50

# Clases y colores (BGR)
CLASS_NAMES  = {0: "knife", 1: "pistol"}
CLASS_COLORS = {"knife": (0, 165, 255), "pistol": (0, 0, 255)}

# API Gen AI  — Groq
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
GROQ_MODEL   = "meta-llama/llama-4-scout-17b-16e-instruct"

# Device
if torch.cuda.is_available():
    DEVICE = 0
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device  : {DEVICE}")
print(f"Modelo  : {MODEL_PATH}")


Device  : 0
Modelo  : /content/best.pt


## 2. Cargar modelo

In [52]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el modelo en:\n  {MODEL_PATH}\n"
        "Asegúrate de haber entrenado en 01_eda.ipynb o descargado best.pt desde Kaggle."
    )

model = YOLO(str(MODEL_PATH))
print(f"Modelo cargado correctamente: {MODEL_PATH.name}")


Modelo cargado correctamente: best.pt


## 3. Funciones del sistema

In [53]:
# ── Registro de sesión (métricas) ────────────────────────────────────────────
session_log: list[dict] = []


def draw_detections(frame: np.ndarray, results, conf_threshold: float) -> tuple[np.ndarray, list[dict]]:
    """Dibuja bboxes sobre el frame y devuelve lista de detecciones."""
    detections = []
    annotated  = frame.copy()
    boxes      = results[0].boxes

    if boxes is None or len(boxes) == 0:
        return annotated, detections

    for box in boxes:
        conf = float(box.conf[0])
        if conf < conf_threshold:
            continue
        cls_id   = int(box.cls[0])
        cls_name = CLASS_NAMES.get(cls_id, f"clase_{cls_id}")
        color    = CLASS_COLORS.get(cls_name, (255, 255, 255))
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        label = f"{cls_name} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(annotated, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(annotated, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        detections.append({
            "class_name": cls_name,
            "confidence": round(conf, 3),
            "bbox": [x1, y1, x2, y2]
        })

    return annotated, detections


def encode_frame_b64(frame_bgr: np.ndarray) -> str:
    """Convierte frame BGR a base64 JPEG."""
    _, buf = cv2.imencode(".jpg", frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return base64.b64encode(buf).decode("utf-8")


def build_prompt(detections: list[dict]) -> str:
    """Construye el prompt para Gen AI."""
    if not detections:
        return (
            "Analiza esta imagen de videovigilancia. No se detectaron armas "
            "automáticamente. Describe la escena brevemente e indica si observas "
            "algún riesgo visual."
        )
    det_text = "\n".join(
        f"  - {d['class_name']} (confianza {d['confidence']:.0%}, "
        f"bbox [{d['bbox'][0]},{d['bbox'][1]}]→[{d['bbox'][2]},{d['bbox'][3]}])"
        for d in detections
    )
    return f"""Eres un sistema de análisis de seguridad para videovigilancia.

Armas detectadas automáticamente:
{det_text}

Proporciona en español:
1. Tipo de arma y descripción visual.
2. Posición en la escena y quién la porta (si es visible).
3. Contexto estimado del entorno.
4. Nivel de riesgo: Bajo / Medio / Alto con justificación breve.
5. Recomendación al operador de seguridad.

Máximo 150 palabras. Respuesta directa, sin encabezados innecesarios.
"""


def describe_event_groq(frame_bgr: np.ndarray, detections: list[dict]) -> str:
    """Llama a la API de Groq con visión y devuelve la descripción."""
    api_key = globals().get("GROQ_API_KEY", "") or os.environ.get("GROQ_API_KEY", "")
    if not api_key:
        return "⚠️ GROQ_API_KEY no configurada. Define la variable de entorno o el secret de Colab."

    img_b64 = encode_frame_b64(frame_bgr)
    prompt  = build_prompt(detections)

    try:
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }],
            max_tokens=400,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error al llamar a Groq: {e}"

print("Funciones del sistema definidas.")


Funciones del sistema definidas.


## 4. Handlers de la interfaz

In [54]:
def handle_image(image_np: np.ndarray, conf: float, use_genai: bool) -> tuple:
    if image_np is None:
        return None, "No se recibió imagen.", ""

    frame_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    # imgsz=320 agregado
    results = model(frame_bgr, conf=conf, device=DEVICE, imgsz=320, verbose=False)
    annotated_bgr, detections = draw_detections(frame_bgr, results, conf)

    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

    if detections:
        rows = "| Clase | Confianza | BBox |\n|---|---|---|\n"
        rows += "\n".join(
            f"| {d['class_name']} | {d['confidence']:.0%} | {d['bbox']} |"
            for d in detections
        )
        tabla = rows
    else:
        tabla = "✅ No se detectaron armas en la imagen."

    session_log.append({
        "tipo": "imagen",
        "timestamp": time.strftime("%H:%M:%S"),
        "detecciones": detections
    })

    descripcion = ""
    if use_genai:
        descripcion = describe_event_groq(annotated_bgr, detections)

    return annotated_rgb, tabla, descripcion


def handle_video(video_path: str, conf: float, frame_skip: int, use_genai: bool) -> tuple:
    if video_path is None:
        return None, "No se recibió video.", ""

    cap    = cv2.VideoCapture(video_path)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Resolución de salida reducida a 640p
    OUT_W = 640
    OUT_H = int(height * 640 / width)

    tmp_out  = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    tmp_h264 = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
    out      = cv2.VideoWriter(tmp_out.name, fourcc, fps, (OUT_W, OUT_H))

    frame_idx       = 0
    all_detections  = []
    best_frame      = None
    best_frame_dets = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_skip == 0:
            # Redimensionar antes de procesar
            frame = cv2.resize(frame, (OUT_W, OUT_H))
            results = model(frame, conf=conf, device=DEVICE, imgsz=320, verbose=False)
            annotated, detections = draw_detections(frame, results, conf)
            all_detections.extend(detections)

            if len(detections) > len(best_frame_dets):
                best_frame_dets = detections
                best_frame = annotated  # sin .copy() innecesario
        else:
            annotated = cv2.resize(frame, (OUT_W, OUT_H))

        out.write(annotated)
        frame_idx += 1

    cap.release()
    out.release()

    # Re-encodear a H.264 directamente — evita el doble encodeo de Gradio
    os.system(
        f'ffmpeg -y -i "{tmp_out.name}" -vcodec libx264 -crf 23 '
        f'"{tmp_h264.name}" -loglevel error'
    )
    video_final = tmp_h264.name if Path(tmp_h264.name).stat().st_size > 0 else tmp_out.name

    clases = {}
    for d in all_detections:
        clases[d["class_name"]] = clases.get(d["class_name"], 0) + 1

    resumen = f"**Frames procesados:** {frame_idx}  \n"
    resumen += f"**Total detecciones:** {len(all_detections)}  \n"
    if clases:
        resumen += "**Por clase:**  \n"
        for cls, cnt in clases.items():
            resumen += f"  - {cls}: {cnt}  \n"
    else:
        resumen += "✅ No se detectaron armas en el video."

    session_log.append({
        "tipo": "video",
        "timestamp": time.strftime("%H:%M:%S"),
        "frames": frame_idx,
        "detecciones": all_detections
    })

    descripcion = ""
    if use_genai and best_frame is not None and best_frame_dets:
        descripcion = describe_event_groq(best_frame, best_frame_dets)

    return video_final, resumen, descripcion


def handle_metrics() -> str:
    if not session_log:
        return "No hay análisis en esta sesión todavía. Sube una imagen o video primero."

    total_img = sum(1 for e in session_log if e["tipo"] == "imagen")
    total_vid = sum(1 for e in session_log if e["tipo"] == "video")
    all_dets  = [d for e in session_log for d in e.get("detecciones", [])]

    clases: dict[str, list[float]] = {}
    for d in all_dets:
        clases.setdefault(d["class_name"], []).append(d["confidence"])

    resumen  = "### 📊 Resumen de sesión\n\n"
    resumen += f"- Imágenes analizadas: **{total_img}**\n"
    resumen += f"- Videos analizados: **{total_vid}**\n"
    resumen += f"- Total detecciones acumuladas: **{len(all_dets)}**\n\n"

    if clases:
        resumen += "#### Detalle por clase\n\n"
        resumen += "| Clase | Detecciones | Confianza promedio | Confianza máx |\n"
        resumen += "|---|---|---|---|\n"
        for cls, confs in clases.items():
            resumen += (
                f"| {cls} | {len(confs)} "
                f"| {sum(confs)/len(confs):.1%} "
                f"| {max(confs):.1%} |\n"
            )
    else:
        resumen += "No se detectaron armas en ningún análisis."

    return resumen


print("Handlers definidos.")

Handlers definidos.


## 5. Interfaz Gradio

Ejecuta esta celda para lanzar la app.
- **Colab:** Se genera un enlace público temporal (`share=True`).
- **Local:** Se abre en `http://localhost:7860`.


In [55]:
# Configurar API key desde Colab Secrets si aplica
if IN_COLAB:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
        print("GROQ_API_KEY cargada desde Colab Secrets.")
    except Exception:
        print("GROQ_API_KEY no encontrada en Secrets — Gen AI desactivado.")

# ── Tema ──────────────────────────────────────────────────────────────────────
THEME = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="purple",
    neutral_hue="slate",
)

# ── UI ────────────────────────────────────────────────────────────────────────
with gr.Blocks(theme=THEME, title="Weapon Detection System") as demo:

    gr.Markdown(
        """
        <img src="https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg" width=220 style="margin-bottom:8px"/>

        # Weapon Detection System
        **Detección de armas en imágenes y video con descripción automática por IA**
        Maestría en Inteligencia Artificial · Universidad ICESI · Proyecto de Innovación III
        ---
        """
    )

    # ── Pestaña 1: Imagen ─────────────────────────────────────────────────────
    with gr.Tab("🖼️ Imagen"):
        gr.Markdown("### Sube una imagen para detectar armas")
        with gr.Row():
            with gr.Column(scale=1):
                img_input = gr.Image(label="Imagen de entrada", type="numpy", sources=["upload"])
                conf_slider = gr.Slider(0.1, 1.0, value=DEFAULT_CONF, step=0.06,
                                        label="Umbral de confianza")
                genai_chk   = gr.Checkbox(label="Generar descripción con IA", value=True)
                img_btn     = gr.Button("🔍 Analizar imagen", variant="primary")
            with gr.Column(scale=1):
                img_output  = gr.Image(label="Resultado", type="numpy")
                img_tabla   = gr.Markdown(label="Detecciones")
                img_desc    = gr.Textbox(label="Descripción del evento (Gen AI)",
                                         lines=6, interactive=False)
        img_btn.click(
            fn=handle_image,
            inputs=[img_input, conf_slider, genai_chk],
            outputs=[img_output, img_tabla, img_desc]
        )

    # ── Pestaña 2: Video ──────────────────────────────────────────────────────
    with gr.Tab("🎬 Video"):
        gr.Markdown("### Sube un video para procesar frame a frame")
        with gr.Row():
            with gr.Column(scale=1):
                vid_input   = gr.Video(label="Video de entrada")
                conf_vid    = gr.Slider(0.1, 1.0, value=DEFAULT_CONF, step=0.06,
                                        label="Umbral de confianza")
                skip_slider = gr.Slider(1, 10, value=2, step=1,
                                        label="Procesar 1 de cada N frames (mayor = más rápido)")
                genai_vid   = gr.Checkbox(label="Descripción Gen AI del frame clave", value=True)
                vid_btn     = gr.Button("▶️ Procesar video", variant="primary")
            with gr.Column(scale=1):
                vid_output  = gr.Video(
                    label="Video procesado",
                    autoplay=True,
                    show_download_button=False
                )
                vid_resumen = gr.Markdown(label="Resumen de detecciones")
                vid_desc    = gr.Textbox(label="Descripción Gen AI (frame con más detecciones)",
                                          lines=6, interactive=False)
        vid_btn.click(
            fn=handle_video,
            inputs=[vid_input, conf_vid, skip_slider, genai_vid],
            outputs=[vid_output, vid_resumen, vid_desc]
        )

    # ── Pestaña 3: Métricas ───────────────────────────────────────────────────
    with gr.Tab("📊 Métricas de sesión"):
        gr.Markdown("""
        ### Resumen de todos los análisis realizados en esta sesión
        Las métricas se acumulan mientras la app está corriendo.
        """)
        metrics_btn    = gr.Button("🔄 Actualizar métricas", variant="secondary")
        metrics_output = gr.Markdown()
        metrics_btn.click(fn=handle_metrics, inputs=[], outputs=[metrics_output])

    gr.Markdown("""
    ---
    Maestría en Inteligencia Artificial — Universidad ICESI
    Modelo: YOLO | Gen AI: Groq (LLaMA 4 Scout)
    """)

# ── Lanzar ────────────────────────────────────────────────────────────────────
demo.launch(
    share=IN_COLAB,
    show_error=True,
    server_name="0.0.0.0" if not IN_COLAB else None
)


GROQ_API_KEY cargada desde Colab Secrets.


/tmp/ipykernel_1043/3558570958.py:18: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=THEME, title="Weapon Detection System") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a50016188accc5c78c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. Detener la app

Ejecuta esta celda para cerrar el servidor Gradio cuando termines.


In [56]:
# demo.close()
# print("App cerrada.")
